# Notebook 18 — NSV Stage 1: Dynamics-Predictive Encoder-Decoder (BSISO MJJAS)
**Project:** ENSO-BSISO SSL — Neural State Variables extension  
**Author:** Jiayi (jh9141@nyu.edu)

**Stage 1 of the NSV pipeline** (Chen et al., arXiv:2112.10755, §2). Train a CNN encoder `g_E` + decoder `g_D` to predict tomorrow's atmospheric field from today's:

$$\mathcal{L}_1 = \mathbb{E}_{(X_t, X_{t+1})}\bigl[\,\lVert g_D(g_E(X_t)) - X_{t+1} \rVert^2\,\bigr]$$

The bottleneck `z = g_E(X_t) ∈ ℝ^{LD}` is **intentionally overparameterized** (`LD=64`, much larger than the expected true ID of 2–4). The encoder is forced to encode enough state information to predict the next day; by the manifold hypothesis the learned latent vectors `{z_t}` then lie on a low-D manifold whose dimension equals the true system ID. nb19 estimates that dimension via Levina-Bickel maximum-likelihood on `{z_t}`.

Why overparameterize? Chen et al. Fig. 5A — directly training a bottleneck of dimension = true ID fails to converge. The two-stage approach (overparameterize → estimate ID → compress with SIREN refine in nb20) bypasses the optimization difficulty.

## Inputs (from nb17 at `BSISO_SSL_Project/nsv/data/`)

- `X_t.npy`, `X_t1.npy` shape `(6536, 3, 31, 51)` — the consecutive-day pair tensors
- `train_mask.npy` shape `(6536,)` bool — year-based split (5,168 train / 1,368 val)
- ancillary label arrays — passed through to nb19/nb20 but not used here

## Architecture (Session 26 §5)

**Encoder `g_E`: (3, 31, 51) → z ∈ ℝ^{64}**

| Block | Output | Notes |
|---|---|---|
| Conv2d(3→32, k=4, s=2, p=1) + BN + ReLU | (32, 15, 25) | stride-2 |
| Conv2d(32→32, k=3, p=1) + BN + ReLU | (32, 15, 25) | refinement |
| Conv2d(32→64, k=4, s=2, p=1) + BN + ReLU | (64, 7, 12) | stride-2 |
| Conv2d(64→64, k=3, p=1) + BN + ReLU | (64, 7, 12) | refinement |
| Conv2d(64→128, k=4, s=2, p=1) + BN + ReLU | (128, 3, 6) | stride-2 |
| AdaptiveAvgPool2d(1) + Flatten + Linear(128, 64) | ℝ^{64} | bottleneck `z_t` |

**Decoder `g_D`: z ∈ ℝ^{64} → (3, 31, 51)**

Uses bilinear `F.interpolate` (not `ConvTranspose2d`) so the decoder hits exact non-power-of-2 sizes (31, 51) without checkerboard artifacts:

| Block | Output |
|---|---|
| Linear(64, 128) + reshape | (128, 1, 1) |
| interpolate(3, 6)   → Conv(128→64,  k=3) + BN + ReLU | (64, 3, 6) |
| interpolate(7, 12)  → Conv(64→64,   k=3) + BN + ReLU | (64, 7, 12) |
| interpolate(15, 25) → Conv(64→32,   k=3) + BN + ReLU | (32, 15, 25) |
| interpolate(31, 51) → Conv(32→3,    k=3)             | (3, 31, 51) — no activation |

Total params ≈ **230 K** (encoder ~140 K, decoder ~90 K). Small enough for a Colab T4.

## Training

| Hyperparameter | Value | Rationale |
|---|---|---|
| Optimizer | Adam, lr=1e-3 | standard for CNNs |
| LR schedule | CosineAnnealingLR, T_max=100 | smooth decay to lr*0.01 |
| Epochs | 100 | conservative; small N, small model |
| Batch size | 64 | fits T4 comfortably |
| Loss | MSE (L2 on normalized fields) | NSV paper convention |
| Weight decay | 1e-4 | light regularization |
| Dropout | none in encoder | encoder must produce deterministic z in eval |

## Outputs (`BSISO_SSL_Project/nsv/`)

- `checkpoints/encoder_stage1.pth`, `checkpoints/decoder_stage1.pth` — final weights
- `checkpoints/training_history_stage1.json` — train/val MSE per epoch
- `latents/z_train.npy` shape `(5168, 64)` — latent vectors for all train pairs
- `latents/z_val.npy` shape `(1368, 64)` — latent vectors for all val pairs
- `results/stage1/training_curves.png` — train/val MSE + per-epoch time
- `results/stage1/reconstructions.png` — 4 val pairs: X_t / X_t1 target / X̂_t1 predicted
- `results/stage1/latent_diagnostics.png` — per-dimension std (should not be all-zero except a few)
- `results/stage1/stage1_summary.md` — final losses + latent diagnostics + decision

## Verification gate (must pass before nb19)

1. **Convergence.** Final train MSE < persistence baseline (~0.5–1.0 for normalized fields). Val MSE should track train MSE without exploding.
2. **Reconstruction quality.** The 4 predicted X̂_t+1 panels should look spatially similar to the X_t+1 targets — coherent large-scale patterns, not blurry constants.
3. **Non-degenerate latent.** `z_train.std(axis=0).min() > 0.01` — no dimension fully collapsed. If many dims are near zero, the effective ID at Stage 1 is already very low.

## Runtime

~10–15 min on Colab T4 (5,168 train pairs × 100 epochs, ~80 batches/epoch, ~50 ms/batch).

---

## Cell 1 — Mount Drive, Load nb17 Outputs, Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

# --- paths ---
PROJECT_DIR  = '/content/drive/MyDrive/BSISO_SSL_Project'
NSV_DIR      = f'{PROJECT_DIR}/nsv'
DATA_DIR     = f'{NSV_DIR}/data'
CKPT_DIR     = f'{NSV_DIR}/checkpoints'
LATENT_DIR   = f'{NSV_DIR}/latents'
RESULTS_DIR  = f'{NSV_DIR}/results/stage1'
for d in [CKPT_DIR, LATENT_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

# --- hyperparameters ---
LATENT_DIM    = 64        # overparameterized bottleneck (true ID expected 2-4)
BATCH_SIZE    = 64
EPOCHS        = 100
LR            = 1e-3
WEIGHT_DECAY  = 1e-4
SEED          = 42

torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# --- load nb17 outputs ---
X_t        = np.load(f'{DATA_DIR}/X_t.npy')
X_t1       = np.load(f'{DATA_DIR}/X_t1.npy')
train_mask = np.load(f'{DATA_DIR}/train_mask.npy')
with open(f'{DATA_DIR}/nsv_data_meta.json') as f:
    meta = json.load(f)

n_total = X_t.shape[0]
n_train = int(train_mask.sum())
n_val   = int((~train_mask).sum())
assert X_t.shape == X_t1.shape == (n_total, 3, 31, 51), f'Unexpected pair shape {X_t.shape}'
print(f'Loaded {n_total} pairs from nb17  (train {n_train} / val {n_val}).')
print(f'X_t range: [{X_t.min():.3f}, {X_t.max():.3f}],  std={X_t.std():.3f}  (normalized field, expect σ≈1)')

## Cell 2 — Encoder + Decoder Architecture

Defines `EncoderBSISO` (5 conv blocks → 64-D bottleneck) and `DecoderBSISO` (bilinear-upsample + conv pyramid → (3, 31, 51)). Sanity-traces shapes through every stage.

In [ ]:
class EncoderBSISO(nn.Module):
    """CNN encoder: (B, 3, 31, 51) -> (B, latent_dim)."""
    def __init__(self, latent_dim=64):
        super().__init__()
        self.conv1 = nn.Conv2d(3,  32,  kernel_size=4, stride=2, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 32,  kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(32)
        self.conv3 = nn.Conv2d(32, 64,  kernel_size=4, stride=2, padding=1, bias=False)
        self.bn3   = nn.BatchNorm2d(64)
        self.conv4 = nn.Conv2d(64, 64,  kernel_size=3, stride=1, padding=1, bias=False)
        self.bn4   = nn.BatchNorm2d(64)
        self.conv5 = nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1, bias=False)
        self.bn5   = nn.BatchNorm2d(128)
        self.gap   = nn.AdaptiveAvgPool2d(1)
        self.fc    = nn.Linear(128, latent_dim)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01); nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        x = F.relu(self.bn4(self.conv4(x)))
        x = F.relu(self.bn5(self.conv5(x)))
        x = self.gap(x).flatten(1)
        return self.fc(x)


class DecoderBSISO(nn.Module):
    """Bilinear-upsample + conv decoder: (B, latent_dim) -> (B, 3, 31, 51)."""
    def __init__(self, latent_dim=64):
        super().__init__()
        self.fc    = nn.Linear(latent_dim, 128)
        self.conv1 = nn.Conv2d(128, 64, kernel_size=3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(64)
        self.conv2 = nn.Conv2d(64,  64, kernel_size=3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64,  32, kernel_size=3, padding=1, bias=False)
        self.bn3   = nn.BatchNorm2d(32)
        self.conv4 = nn.Conv2d(32,  3,  kernel_size=3, padding=1)   # final, no BN, no activation
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None: nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01); nn.init.constant_(m.bias, 0)

    def forward(self, z):
        x = self.fc(z).view(-1, 128, 1, 1)
        x = F.interpolate(x, size=(3, 6),   mode='bilinear', align_corners=False)
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.interpolate(x, size=(7, 12),  mode='bilinear', align_corners=False)
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.interpolate(x, size=(15, 25), mode='bilinear', align_corners=False)
        x = F.relu(self.bn3(self.conv3(x)))
        x = F.interpolate(x, size=(31, 51), mode='bilinear', align_corners=False)
        return self.conv4(x)


# --- sanity check (build, trace shapes, count params) ---
enc = EncoderBSISO(LATENT_DIM).to(device)
dec = DecoderBSISO(LATENT_DIM).to(device)
n_params_enc = sum(p.numel() for p in enc.parameters())
n_params_dec = sum(p.numel() for p in dec.parameters())

with torch.no_grad():
    dummy = torch.randn(4, 3, 31, 51).to(device)
    z = enc(dummy)
    xhat = dec(z)
    print(f'Encoder params: {n_params_enc:,}    Decoder params: {n_params_dec:,}    Total: {n_params_enc + n_params_dec:,}')
    print(f'Forward sanity:  input {tuple(dummy.shape)}  ->  z {tuple(z.shape)}  ->  output {tuple(xhat.shape)}')
    assert z.shape == (4, LATENT_DIM), f'Unexpected z shape {z.shape}'
    assert xhat.shape == dummy.shape, f'Decoder output shape mismatch: {xhat.shape} vs {dummy.shape}'
    print(f'Init MSE (random):  {F.mse_loss(xhat, dummy).item():.4f}')
    print('✓ Architecture sanity-checked.')

## Cell 3 — Dataset + DataLoaders

A tiny `PairDataset` wrapping numpy arrays. The whole pair tensor fits in RAM (~250 MB total), so we keep it as a tensor for max speed.

In [ ]:
class PairDataset(Dataset):
    def __init__(self, X_t, X_t1, indices):
        self.X_t  = torch.from_numpy(X_t[indices]).float()
        self.X_t1 = torch.from_numpy(X_t1[indices]).float()
    def __len__(self):  return self.X_t.shape[0]
    def __getitem__(self, k):  return self.X_t[k], self.X_t1[k]

train_idx = np.where(train_mask)[0]
val_idx   = np.where(~train_mask)[0]

train_ds = PairDataset(X_t, X_t1, train_idx)
val_ds   = PairDataset(X_t, X_t1, val_idx)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds):5d} pairs  -> {len(train_loader)} batches/epoch')
print(f'Val:   {len(val_ds):5d} pairs  -> {len(val_loader)} batches/epoch')

# Persistence baseline on val (predicting X_t1 == X_t) — a sanity floor for the model.
with torch.no_grad():
    persistence_mse = ((val_ds.X_t - val_ds.X_t1) ** 2).mean().item()
print(f'Persistence MSE on val (predict X_t+1 = X_t): {persistence_mse:.4f}  (the model should beat this).')

## Cell 4 — Training Loop (MSE on next-day prediction)

Standard PyTorch train/val loop. Tracks per-epoch train MSE, val MSE, and time. Saves a checkpoint every 25 epochs and at the end.

In [ ]:
from tqdm.notebook import tqdm

params = list(enc.parameters()) + list(dec.parameters())
optimizer = optim.Adam(params, lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR * 0.01)

history = {'train_mse': [], 'val_mse': [], 'epoch_time': []}
best_val = float('inf')

for epoch in range(EPOCHS):
    t0 = time.time()

    enc.train(); dec.train()
    train_loss = 0.0; n_train_seen = 0
    pbar = tqdm(train_loader, desc=f'ep {epoch+1}/{EPOCHS}', leave=False)
    for x_t, x_t1 in pbar:
        x_t  = x_t.to(device, non_blocking=True)
        x_t1 = x_t1.to(device, non_blocking=True)
        z = enc(x_t)
        x_t1_hat = dec(z)
        loss = F.mse_loss(x_t1_hat, x_t1)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        train_loss += loss.item() * x_t.size(0); n_train_seen += x_t.size(0)
        pbar.set_postfix({'mse': f'{loss.item():.4f}'})
    train_mse = train_loss / n_train_seen

    enc.eval(); dec.eval()
    val_loss = 0.0; n_val_seen = 0
    with torch.no_grad():
        for x_t, x_t1 in val_loader:
            x_t  = x_t.to(device, non_blocking=True)
            x_t1 = x_t1.to(device, non_blocking=True)
            val_loss += F.mse_loss(dec(enc(x_t)), x_t1, reduction='sum').item() / x_t1.numel() * x_t.size(0)
            # equivalent: mean MSE * batch_size, then divide by n_val at end
            n_val_seen += x_t.size(0)
    val_mse = val_loss / n_val_seen
    scheduler.step()

    et = time.time() - t0
    history['train_mse'].append(train_mse)
    history['val_mse'].append(val_mse)
    history['epoch_time'].append(et)
    print(f'ep {epoch+1:3d}/{EPOCHS}  train={train_mse:.4f}  val={val_mse:.4f}  '
          f'lr={scheduler.get_last_lr()[0]:.2e}  time={et:.1f}s')

    if val_mse < best_val:
        best_val = val_mse
        torch.save(enc.state_dict(), f'{CKPT_DIR}/encoder_stage1_best.pth')
        torch.save(dec.state_dict(), f'{CKPT_DIR}/decoder_stage1_best.pth')

torch.save(enc.state_dict(), f'{CKPT_DIR}/encoder_stage1.pth')
torch.save(dec.state_dict(), f'{CKPT_DIR}/decoder_stage1.pth')
with open(f'{CKPT_DIR}/training_history_stage1.json', 'w') as f:
    json.dump(history, f, indent=2)

print(f'\nDone in {sum(history["epoch_time"])/60:.1f} min.')
print(f'Best val MSE: {best_val:.4f}  (at epoch {history["val_mse"].index(best_val)+1}).')
print(f'Final train MSE: {history["train_mse"][-1]:.4f}  val MSE: {history["val_mse"][-1]:.4f}')
print(f'Persistence baseline (val): {persistence_mse:.4f}  -> the model {"beats" if best_val < persistence_mse else "does NOT beat"} persistence.')

## Cell 5 — Training Curves + Reconstruction Visualization

Three diagnostics:
1. **Loss curves** — train and val MSE per epoch, plus the persistence baseline as a horizontal reference.
2. **Reconstructions** — 4 random val pairs, OLR channel, three columns: `X_t`, `X_t+1` (target), `X̂_t+1` (predicted). Each row should show three spatially coherent maps.
3. **Latent diagnostics** — per-dimension std across the 64-D bottleneck. Most dims should be active; if many are near zero, the effective ID at Stage 1 is already very low.

In [ ]:
# Load best checkpoint for diagnostics (better val MSE than final)
enc.load_state_dict(torch.load(f'{CKPT_DIR}/encoder_stage1_best.pth', map_location=device))
dec.load_state_dict(torch.load(f'{CKPT_DIR}/decoder_stage1_best.pth', map_location=device))
enc.eval(); dec.eval()

# 1) Training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].plot(history['train_mse'], label='Train MSE', lw=2)
axes[0].plot(history['val_mse'],   label='Val MSE',   lw=2)
axes[0].axhline(persistence_mse, color='gray', ls='--', lw=1, label=f'Persistence ({persistence_mse:.3f})')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE')
axes[0].set_title('Training Curves', fontweight='bold')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history['epoch_time'], color='green', lw=2)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Time (s)')
axes[1].set_title('Per-Epoch Time', fontweight='bold')
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/training_curves.png', dpi=140, bbox_inches='tight')
plt.show()

# 2) Reconstructions on 4 random val pairs (OLR channel = 2)
rng = np.random.default_rng(SEED)
picks = rng.choice(len(val_ds), size=4, replace=False)
x_t_pick  = val_ds.X_t[picks].to(device)
x_t1_pick = val_ds.X_t1[picks].to(device)
with torch.no_grad():
    x_t1_hat = dec(enc(x_t_pick)).cpu().numpy()
x_t_np  = x_t_pick.cpu().numpy()
x_t1_np = x_t1_pick.cpu().numpy()

olr_ch = 2
fig, axes = plt.subplots(4, 3, figsize=(13, 14), sharex=True, sharey=True)
fig.suptitle("Val-set reconstructions (OLR channel): X_t (left) | X_t+1 target (middle) | X̂_t+1 predicted (right)",
             fontsize=12, fontweight='bold')
vmax = max(np.abs(x_t_np[:, olr_ch]).max(), np.abs(x_t1_np[:, olr_ch]).max(), np.abs(x_t1_hat[:, olr_ch]).max())
for r in range(4):
    for c, (arr, label) in enumerate([(x_t_np, 'X_t'), (x_t1_np, 'X_t+1 target'), (x_t1_hat, 'X̂_t+1 predicted')]):
        ax = axes[r, c]
        im = ax.imshow(arr[r, olr_ch], cmap='RdBu_r', aspect='auto',
                       extent=[60, 160, 0, 60], vmin=-vmax, vmax=vmax, origin='lower')
        if r == 0: ax.set_title(label, fontsize=11, fontweight='bold')
        if c == 0: ax.set_ylabel(f'pair {picks[r]}', fontsize=10)
        if r == 3: ax.set_xlabel('Longitude (°)')
fig.subplots_adjust(right=0.90)
cb = fig.add_axes([0.92, 0.15, 0.015, 0.7])
plt.colorbar(im, cax=cb, label="OLR' (σ)")
plt.savefig(f'{RESULTS_DIR}/reconstructions.png', dpi=130, bbox_inches='tight')
plt.show()

# 3) Latent diagnostics — extract z on all train pairs, plot per-dim std
with torch.no_grad():
    z_train_list = []
    for k in range(0, len(train_ds), 256):
        batch = train_ds.X_t[k:k+256].to(device)
        z_train_list.append(enc(batch).cpu().numpy())
    z_train_diag = np.concatenate(z_train_list, axis=0)

z_mean = z_train_diag.mean(axis=0)
z_std  = z_train_diag.std(axis=0)

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(np.arange(LATENT_DIM), z_std, color='steelblue', alpha=0.85)
ax.axhline(z_std.mean(), color='red', ls='--', lw=1, label=f'Mean std = {z_std.mean():.3f}')
ax.axhline(0.01, color='gray', ls=':', lw=1, label='Collapse threshold (0.01)')
ax.set_xlabel('Latent dimension index'); ax.set_ylabel('std across train pairs')
ax.set_title(f'Per-Dimension std (LD={LATENT_DIM}).  Active dims (std > 0.01): {(z_std > 0.01).sum()}/{LATENT_DIM}',
             fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/latent_diagnostics.png', dpi=140, bbox_inches='tight')
plt.show()

print(f'z_train summary:  mean ‖·‖ over dims: μ_mean={z_mean.mean():+.4f}, σ_mean={z_mean.std():.4f}')
print(f'                  per-dim std:        min={z_std.min():.4f}, max={z_std.max():.4f}, mean={z_std.mean():.4f}')
n_active = int((z_std > 0.01).sum())
assert n_active >= LATENT_DIM // 4, f'Only {n_active}/{LATENT_DIM} active dims — likely rank collapse.'
print(f'✓ {n_active}/{LATENT_DIM} latent dims active (std > 0.01).')

## Cell 6 — Extract and Save Latent Vectors

Run the encoder (`eval` mode, no grad) on the **full** train and val pair sets, save `z_train.npy` and `z_val.npy`. These are the inputs to nb19 (Levina-Bickel ID estimation).

In [ ]:
def extract_z(dataset, encoder, batch=256):
    encoder.eval()
    zs = []
    with torch.no_grad():
        for k in range(0, len(dataset), batch):
            x = dataset.X_t[k:k+batch].to(device)
            zs.append(encoder(x).cpu().numpy())
    return np.concatenate(zs, axis=0).astype(np.float32)

z_train = extract_z(train_ds, enc)
z_val   = extract_z(val_ds,   enc)

np.save(f'{LATENT_DIR}/z_train.npy', z_train)
np.save(f'{LATENT_DIR}/z_val.npy',   z_val)

print(f'Saved latents:')
print(f'  z_train.npy  shape {z_train.shape}  range [{z_train.min():.3f}, {z_train.max():.3f}]')
print(f'  z_val.npy    shape {z_val.shape}    range [{z_val.min():.3f}, {z_val.max():.3f}]')

# Final summary file
import time as _t
stage1_summary = {
    'date':                _t.strftime('%Y-%m-%d'),
    'latent_dim':          LATENT_DIM,
    'epochs':              EPOCHS,
    'best_val_mse':        float(best_val),
    'final_train_mse':     float(history['train_mse'][-1]),
    'final_val_mse':       float(history['val_mse'][-1]),
    'persistence_mse':     float(persistence_mse),
    'beats_persistence':   bool(best_val < persistence_mse),
    'n_active_latent_dims': int((z_std > 0.01).sum()),
    'z_train_shape':       list(z_train.shape),
    'z_val_shape':         list(z_val.shape),
    'meta_from_nb17':      meta,
}
with open(f'{RESULTS_DIR}/stage1_summary.json', 'w') as f:
    json.dump(stage1_summary, f, indent=2)
print(f'\nSaved summary: {RESULTS_DIR}/stage1_summary.json')
print('\n✓ Stage 1 complete. Proceed to nb19 (intrinsic dimension estimation).')

---
## Done!

**Send back** for review:
1. The printed output of Cell 4 (training loop) — final train/val MSE + persistence comparison.
2. `results/stage1/training_curves.png` — should show train and val MSE converging below the persistence baseline.
3. `results/stage1/reconstructions.png` — the predicted X̂_t+1 should look spatially similar to the X_t+1 target (large-scale BSISO patterns recovered).
4. `results/stage1/latent_diagnostics.png` — the bar chart should show most of the 64 dims with non-trivial std (collapse would show only a handful of active dims).
5. `results/stage1/stage1_summary.json` — top-line summary numbers.

**Decision points for nb19:**
- If `n_active_latent_dims` is, say, 30+ and `beats_persistence = True`: clean Stage 1 — proceed to nb19 to estimate the true ID via Levina-Bickel.
- If `n_active_latent_dims < 16` already: the encoder has implicitly compressed below 64. ID estimate from nb19 will be small (good — that's the point). Still proceed.
- If `beats_persistence = False`: the encoder didn't learn meaningful dynamics. nb19's ID estimate may be meaningless. Inspect training curves and reconstruction quality before deciding to rerun with more epochs / smaller LR.

---
*DDCS Project | jh9141@nyu.edu*